In [ ]:
print("hello world")

In [3]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [ ]:
from unsloth import FastLanguageModel

In [ ]:
import re
import json

def process_mixed_srt(file_path):
    """
    专门处理中英双语混排的 SRT 文件
    """
    with open(file_path, 'r', encoding='utf-8') as f:
        content = f.read()

    # 1. 按照 SRT 序号切分块
    # 匹配模式：数字序号 + 时间轴
    blocks = re.split(r'\n\d+\n\d{2}:\d{2}:\d{2},\d{3} --> \d{2}:\d{2}:\d{2},\d{3}\n', content)

    refined_data = []

    for block in blocks:
        lines = [l.strip() for l in block.split('\n') if l.strip()]

        # 逻辑：如果一个块里有两行，通常第一行是中文，第二行是英文
        # 或者使用正则表达式判断中英
        if len(lines) >= 2:
            # 简单的正则：判断哪行包含中文
            zh_line = ""
            en_line = ""
            for l in lines:
                if re.search(r'[\u4e00-\u9fa5]', l): # 包含汉字
                    zh_line = l
                else:
                    en_line = l

            if zh_line and en_line:
                refined_data.append({"en": en_line, "zh": zh_line})

        # 如果只有一行（比如第1个块的 Joyner），在这个场景下通常是名字或语气词，建议过滤或跳过
        elif len(lines) == 1:
            # 如果你想保留单行作为背景，可以取消下面注释
            # refined_data.append({"en": lines[0], "zh": lines[0]})
            pass

    return refined_data

def convert_to_sharegpt_custom(pairs, output_file, window_size=5):
    """
    将提取的对齐数据转换为 ShareGPT 格式
    """
    dataset = []
    system_prompt = "你是一位精通美国嘻哈文化和俚语的翻译专家。你的任务是翻译歌词，并对其中的文化梗进行深度解析。"

    for i in range(0, len(pairs), window_size):
        chunk = pairs[i : i + window_size]
        en_text = "\n".join([p["en"] for p in chunk])
        zh_text = "\n".join([p["zh"] for p in chunk])

        sample = {
            "messages": [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": f"歌词原文：\n{en_text}"},
                {
                    "role": "assistant",
                    "content": f"歌词译文：\n{zh_text}"
                }
            ]
        }
        dataset.append(sample)

    with open(output_file, 'w', encoding='utf-8') as f:
        for entry in dataset:
            f.write(json.dumps(entry, ensure_ascii=False) + '\n')

    print(f"处理完成！生成了 {len(dataset)} 个训练单元。")

# --- 执行 ---
# 假设你的文件叫 lyrics.srt
# pairs = process_mixed_srt("lyrics.srt")
# convert_to_sharegpt_custom(pairs, "hiphop_expert_train.jsonl")

In [5]:
from datasets import load_dataset

# 加载你的 JSONL 文件
dataset = load_dataset("json", data_files="/content/train_data/hiphop_finetune_data.jsonl", split="train")

# 检查一下数据格式
print(f"✅ 数据加载成功，共 {len(dataset)} 条样本")
print(f"样例数据: {dataset[0]['messages'][1]['content'][:50]}...")

Generating train split: 0 examples [00:00, ? examples/s]

✅ 数据加载成功，共 117 条样本
样例数据: 歌词原文：
As I get a little older, I realize life is p...


In [4]:
from unsloth import FastLanguageModel
import torch

# 1. 加载 Qwen2.5-7B-Instruct
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit",
    max_seq_length = 2048,
    load_in_4bit = True,
)

# 2. 同样的推理测试（看看没训练前 Qwen 的实力）
FastLanguageModel.for_inference(model)

test_lyric = "Life can bring much pain. There are many variations of pain and I'm a connoisseur of all."
messages = [
    {"role": "system", "content": "你是一位精通嘻哈文化和 J. Cole 风格的翻译专家。"},
    {"role": "user", "content": f"翻译并解析歌词: '{test_lyric}'"}
]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    return_tensors = "pt",
).to("cuda")

with torch.no_grad():
    outputs = model.generate(
        input_ids = inputs,
        max_new_tokens = 512,
        temperature = 0.7,
        do_sample = True,
        pad_token_id = tokenizer.pad_token_id
    )

print("--- [Qwen2.5 原始翻译结果] ---")
print(tokenizer.decode(outputs[0], skip_special_tokens = True))

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.2.1: Fast Qwen2 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.55G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

--- [Qwen2.5 原始翻译结果] ---
system
你是一位精通嘻哈文化和 J. Cole 风格的翻译专家。
user
翻译并解析歌词: 'Life can bring much pain. There are many variations of pain and I'm a connoisseur of all.'
assistant
这段歌词出自 J. Cole 的歌曲《20 something》，其中包含了他对生活的深刻理解和对痛苦的广泛体验。让我们逐句解析：

1. **Life can bring much pain.**
   - **直译**：生活可以带来很多痛苦。
   - **解析**：这句话强调了生活中的痛苦是普遍存在的，而不仅仅是某个特定的人或情境所独有的。J. Cole 认为生活本身就是一个充满挑战和痛苦的过程。

2. **There are many variations of pain.**
   - **直译**：有许多种类型的痛苦。
   - **解析**：这里提到的“许多种类型的痛苦”意味着痛苦的形式是多样化的，可能包括身体上的、情感上的、心理上的等等。这表明 J. Cole 认识到不同的人可能会经历不同的痛苦形式，但这些痛苦都是生活的一部分。

3. **I'm a connoisseur of all.**
   - **直译**：我是一个所有类型痛苦的鉴赏家。
   - **解析**：使用“connoisseur（鉴赏家）”这个词表明 J. Cole 对各种类型的痛苦有着深刻的理解和认识。他不仅经历过多种类型的痛苦，而且能够区分和理解每一种痛苦的独特之处。这反映了他对生活的复杂性和多面性的深刻洞察力。

综合起来，这段歌词传达了 J. Cole 对生活及其带来的各种挑战和痛苦的深刻理解与接受态度。他认为，尽管生活中充满了痛苦，但他愿意去体验和理解所有形式的痛苦，这显示了他成熟和坚韧的一面。


In [9]:
from unsloth import standardize_sharegpt
from trl import SFTTrainer
from transformers import TrainingArguments

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,               # LoRA 的秩，值越大拟合能力越强，16 是平衡点
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",], # 针对 Qwen 的全量层优化
    lora_alpha = 16,
    lora_dropout = 0,     # 设置为 0 性能最强
    bias = "none",        # 设置为 none 性能最强
    use_gradient_checkpointing = "unsloth", # 节省显存，T4 必备
    random_state = 3407,
    use_rslora = False,   # 我们先用标准的 LoRA
    loftq_config = None, 
)

print("✅ LoRA 适配器已挂载，现在可以开始微调了！")

# 然后再运行之前的 trainer = SFTTrainer(...) 定义部分

# 1. 准备数据（Qwen 也支持 standardize_sharegpt）
dataset = standardize_sharegpt(dataset)

def formatting_prompts_func(examples):
    instructions = examples["messages"]
    # 使用 Qwen 的模板
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in instructions]
    return { "text" : texts, }

dataset = dataset.map(formatting_prompts_func, batched=True)

# 2. 训练配置（既然基座很强，我们可以用较小的学习率）
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = 2048,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        max_steps = 60,         # Qwen 学得快，60步先看看效果
        learning_rate = 5e-5,   # 稍微调低，保护它原有的中文智商
        fp16 = True,
        optim = "adamw_8bit",
        logging_steps = 1,
        output_dir = "qwen_jcole_outputs",
    ),
)

trainer.train()

Unsloth 2026.2.1 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


✅ LoRA 适配器已挂载，现在可以开始微调了！


Map:   0%|          | 0/117 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/117 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 117 | Num Epochs = 4 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 40,370,176 of 7,655,986,688 (0.53% trained)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:wandb: You chose "Don't visualize my results"
wandb: Using W&B in offline mode.
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/


Step,Training Loss
1,3.739100
2,4.107300
3,3.894100
4,3.659200
5,3.550900
6,3.651900
7,3.476500
8,3.609700
9,3.660700
10,3.388200


wandb: WARNING URL not available in offline run


train/epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▅▆▆▆▆▇▇▇▇▇████
train/global_step,▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇█
train/grad_norm,█▇▆▆▅▅▅▅▆▆▆▆▆▇▇█▆▆▆▇▅▄▃▃▃▂▁▁▁▂▂▂▁▁▁▁▁▁▁▃
train/learning_rate,████▇▇▇▇▇▆▆▆▆▆▆▅▅▅▅▅▅▅▄▄▄▄▄▃▃▃▃▃▃▃▂▂▂▂▁▁
train/loss,▇█▇▇▆▆▆▅▄▄▄▄▄▄▄▃▃▃▃▂▃▂▃▃▃▃▂▂▂▃▃▄▂▃▂▃▁▂▂▂
total_flos,3879869544456192.0
train/epoch,4
train/global_step,60
train/grad_norm,0.60527
train/learning_rate,0.0
train/loss,2.4425


TrainOutput(global_step=60, training_loss=2.774261369307836, metrics={'train_runtime': 293.3584, 'train_samples_per_second': 1.636, 'train_steps_per_second': 0.205, 'total_flos': 3879869544456192.0, 'train_loss': 2.774261369307836, 'epoch': 4.0})

In [11]:
FastLanguageModel.for_inference(model)

# 2. 准备 Prompt
test_quote = "No such thing as a life that's better than yours."
messages = [
    {"role": "user", "content": f"翻译 J. Cole 在《Love Yourz》里的这句名言并深度解析：'{test_quote}'"}
]

# 3. 编码输入并手动生成 Attention Mask
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    return_tensors = "pt",
).to("cuda")

# 4. 显式设置生成参数，消除警告
outputs = model.generate(
    input_ids = inputs,
    attention_mask = (inputs != tokenizer.pad_token_id).long(), # 显式设置 Mask
    max_new_tokens = 512,
    temperature = 0.5,      # 稍微降低随机性，让解析更深沉
    repetition_penalty = 1.2,
    do_sample = True,
    pad_token_id = tokenizer.eos_token_id
)

print("--- [微调后的 J. Cole 专家模型最终成果] ---")
final_output = tokenizer.decode(outputs[0], skip_special_tokens = True)
# 过滤掉 Prompt 部分，只看 Assistant 的回答
print(final_output.split("assistant")[-1].strip())

--- [微调后的 J. Cole 专家模型最终成果] ---
J. Cole在歌曲《Love Yourz》中的这句话是：“There ain't no such thing as a life that’s better than yours.” 这句话的中文译文可以理解为“不存在比你的生活更好的人生”。

### 深度解析：

1. **平等性**：
   - 该歌词强调每个人的生活都是独一无二且有价值的。它挑战了社会中常有的等级观念和比较心态，提醒人们不要因为看到别人看似更成功或更有成就就感到自卑。
   
2. **自我接纳与自尊心**：
   - 歌词鼓励听众接受自己的现状，并认识到自己所经历的一切都有其价值所在。这种观点有助于增强个人的身份认同感以及自信。

3. **积极面对现实**：
   - 它传达了一种正面的心态——即使当前处境并不理想或者充满困难，也要相信未来会更好；同时珍惜当下拥有的一切美好事物。
   
4. **批判社会不公现象**：
   - 对于那些通过他人对比来评价自身价值的人来说，这首歌提供了一个有力反驳的机会。“没有比你过得好的人”这一理念是对那种基于外部标准评判生活方式正当性的批评。
   
5. **促进团结和支持精神**：
   - 当个体意识到所有人的旅程都不同寻常时，则更容易对他人的困境表示同情而不是嫉妒；从而加强社区内部之间的联系纽带。
    
总之，“there ain’t no such thing as a life that’s better than yours”不仅是一条鼓舞人心的信息，也是对现代社会普遍存在的焦虑情绪的一种疗愈剂，在帮助我们更好地理解和欣赏各自独特的人生体验方面发挥了重要作用。
